## 1. Clone GitHub Repository

In [ ]:
# Clone repository từ GitHub
# Thay YOUR_USERNAME và YOUR_REPO bằng username và tên repo của bạn
!git clone https://github.com/YOUR_USERNAME/YOUR_REPO.git
%cd YOUR_REPO/Lab5

## 2. Install Dependencies

In [ ]:
!pip install -q torch torchvision torchaudio
!pip install -q scikit-learn tqdm

## 3. Import Libraries

In [ ]:
import sys
import os
import torch
import json

# Add src to path
sys.path.append('src')

from models.transformer_encoder import (
    TransformerForSequenceClassification,
    TransformerForTokenClassification
)
from data.uit_viocd_dataset import create_uit_viocd_dataloaders
from data.phonert_dataset import create_phonert_dataloaders
from training.trainer import create_trainer
from utils.utils import set_seed, get_device, print_model_info

# Set seed cho reproducibility
set_seed(42)

# Get device
device = get_device()

## 4. Verify Data Paths

In [ ]:
# Check data directories
print("UIT-ViOCD data files:")
!ls -lh src/data/UIT_ViOCD/

print("\nPhoNERT data files:")
!ls -lh src/data/PhoNERT/

---
# BÀI 1: PHÂN LOẠI DOMAIN - UIT-ViOCD

Xây dựng mô hình Transformer Encoder 3 lớp cho bài toán phân loại domain câu bình luận.

## 4.1. Load UIT-ViOCD Dataset

In [ ]:
# Paths
train_path_viocd = 'src/data/UIT_ViOCD/train_preprocessed.json'
dev_path_viocd = 'src/data/UIT_ViOCD/dev_preprocessed.json'
test_path_viocd = 'src/data/UIT_ViOCD/test_preprocessed.json'

# Hyperparameters
BATCH_SIZE = 32
MAX_LEN = 128
D_MODEL = 256
NUM_HEADS = 8
NUM_LAYERS = 3
D_FF = 1024
DROPOUT = 0.1
LEARNING_RATE = 1e-4
NUM_EPOCHS = 20
PATIENCE = 5

# Create dataloaders
print("Creating UIT-ViOCD dataloaders...")
train_loader_viocd, dev_loader_viocd, test_loader_viocd, vocab_viocd, num_classes = create_uit_viocd_dataloaders(
    train_path_viocd, dev_path_viocd, test_path_viocd,
    batch_size=BATCH_SIZE,
    max_len=MAX_LEN,
    num_workers=2
)

print(f"\nVocabulary size: {len(vocab_viocd)}")
print(f"Number of classes: {num_classes}")
print(f"Train batches: {len(train_loader_viocd)}")
print(f"Dev batches: {len(dev_loader_viocd)}")
print(f"Test batches: {len(test_loader_viocd)}")

## 4.2. Build Model cho Bài 1

In [ ]:
# Create model
model_viocd = TransformerForSequenceClassification(
    vocab_size=len(vocab_viocd),
    num_classes=num_classes,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    d_ff=D_FF,
    max_len=MAX_LEN,
    dropout=DROPOUT,
    pad_idx=vocab_viocd.PAD_IDX
)

print_model_info(model_viocd)

## 4.3. Train Model cho Bài 1

In [ ]:
# Create trainer
trainer_viocd = create_trainer(
    model=model_viocd,
    train_loader=train_loader_viocd,
    dev_loader=dev_loader_viocd,
    test_loader=test_loader_viocd,
    learning_rate=LEARNING_RATE,
    device=device,
    save_dir='checkpoints_viocd',
    task_type='classification'
)

# Train
trainer_viocd.train(num_epochs=NUM_EPOCHS, patience=PATIENCE)

## 4.4. Test Model cho Bài 1

In [ ]:
# Test
test_results_viocd = trainer_viocd.test()

## 4.5. Save Results cho Bài 1

In [ ]:
# Save results
results_bai1 = {
    'task': 'UIT-ViOCD Domain Classification',
    'vocab_size': len(vocab_viocd),
    'num_classes': num_classes,
    'model_parameters': {
        'd_model': D_MODEL,
        'num_heads': NUM_HEADS,
        'num_layers': NUM_LAYERS,
        'd_ff': D_FF,
        'dropout': DROPOUT
    },
    'best_epoch': trainer_viocd.best_epoch,
    'best_dev_accuracy': float(trainer_viocd.best_dev_metric),
    'test_loss': float(test_results_viocd[0]),
    'test_accuracy': float(test_results_viocd[1]),
    'test_precision': float(test_results_viocd[2]),
    'test_recall': float(test_results_viocd[3]),
    'test_f1': float(test_results_viocd[4])
}

with open('results_bai1.json', 'w', encoding='utf-8') as f:
    json.dump(results_bai1, f, indent=2, ensure_ascii=False)

print("\nResults saved to results_bai1.json")
print(json.dumps(results_bai1, indent=2, ensure_ascii=False))

---
# BÀI 2: GÁN NHÃN CHUỖI (NER) - PhoNERT

Xây dựng mô hình Transformer Encoder 3 lớp cho bài toán gán nhãn chuỗi (Named Entity Recognition).

## 5.1. Load PhoNERT Dataset

In [ ]:
# Paths
train_path_phonert = 'src/data/PhoNERT/train.json'
dev_path_phonert = 'src/data/PhoNERT/dev.json'
test_path_phonert = 'src/data/PhoNERT/test.json'

# Hyperparameters (giữ nguyên như Bài 1)
# Có thể điều chỉnh nếu cần

# Create dataloaders
print("Creating PhoNERT dataloaders...")
train_loader_phonert, dev_loader_phonert, test_loader_phonert, vocab_phonert, label_encoder, num_labels = create_phonert_dataloaders(
    train_path_phonert, dev_path_phonert, test_path_phonert,
    batch_size=BATCH_SIZE,
    max_len=MAX_LEN,
    num_workers=2
)

print(f"\nVocabulary size: {len(vocab_phonert)}")
print(f"Number of labels: {num_labels}")
print(f"Label mapping: {label_encoder.label2idx}")
print(f"Train batches: {len(train_loader_phonert)}")
print(f"Dev batches: {len(dev_loader_phonert)}")
print(f"Test batches: {len(test_loader_phonert)}")

## 5.2. Build Model cho Bài 2

In [ ]:
# Create model
model_phonert = TransformerForTokenClassification(
    vocab_size=len(vocab_phonert),
    num_labels=num_labels,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    d_ff=D_FF,
    max_len=MAX_LEN,
    dropout=DROPOUT,
    pad_idx=vocab_phonert.PAD_IDX
)

print_model_info(model_phonert)

## 5.3. Train Model cho Bài 2

In [ ]:
# Create trainer
trainer_phonert = create_trainer(
    model=model_phonert,
    train_loader=train_loader_phonert,
    dev_loader=dev_loader_phonert,
    test_loader=test_loader_phonert,
    learning_rate=LEARNING_RATE,
    device=device,
    save_dir='checkpoints_phonert',
    task_type='token_classification'
)

# Train
trainer_phonert.train(num_epochs=NUM_EPOCHS, patience=PATIENCE)

## 5.4. Test Model cho Bài 2

In [ ]:
# Test
test_results_phonert = trainer_phonert.test()

## 5.5. Save Results cho Bài 2

In [ ]:
# Save results
results_bai2 = {
    'task': 'PhoNERT Named Entity Recognition',
    'vocab_size': len(vocab_phonert),
    'num_labels': num_labels,
    'label_mapping': label_encoder.label2idx,
    'model_parameters': {
        'd_model': D_MODEL,
        'num_heads': NUM_HEADS,
        'num_layers': NUM_LAYERS,
        'd_ff': D_FF,
        'dropout': DROPOUT
    },
    'best_epoch': trainer_phonert.best_epoch,
    'best_dev_f1': float(trainer_phonert.best_dev_metric),
    'test_loss': float(test_results_phonert[0]),
    'test_f1': float(test_results_phonert[1]),
    'test_precision': float(test_results_phonert[2]),
    'test_recall': float(test_results_phonert[3])
}

with open('results_bai2.json', 'w', encoding='utf-8') as f:
    json.dump(results_bai2, f, indent=2, ensure_ascii=False)

print("\nResults saved to results_bai2.json")
print(json.dumps(results_bai2, indent=2, ensure_ascii=False))

---
# 6. SUMMARY

## Tổng kết kết quả

In [ ]:
print("="*70)
print(" "*20 + "FINAL RESULTS")
print("="*70)

print("\n" + "="*70)
print("BÀI 1: UIT-ViOCD Domain Classification")
print("="*70)
print(f"Best Dev Accuracy: {results_bai1['best_dev_accuracy']:.4f} (Epoch {results_bai1['best_epoch']})")
print(f"Test Accuracy:     {results_bai1['test_accuracy']:.4f}")
print(f"Test Precision:    {results_bai1['test_precision']:.4f}")
print(f"Test Recall:       {results_bai1['test_recall']:.4f}")
print(f"Test F1:           {results_bai1['test_f1']:.4f}")

print("\n" + "="*70)
print("BÀI 2: PhoNERT Named Entity Recognition")
print("="*70)
print(f"Best Dev F1:       {results_bai2['best_dev_f1']:.4f} (Epoch {results_bai2['best_epoch']})")
print(f"Test F1:           {results_bai2['test_f1']:.4f}")
print(f"Test Precision:    {results_bai2['test_precision']:.4f}")
print(f"Test Recall:       {results_bai2['test_recall']:.4f}")

print("\n" + "="*70)
print(" "*15 + "TRAINING COMPLETED!")
print("="*70)

## Download Results và Checkpoints

Các file quan trọng:
- `results_bai1.json` - Kết quả Bài 1
- `results_bai2.json` - Kết quả Bài 2
- `checkpoints_viocd/best_model.pt` - Best model Bài 1
- `checkpoints_phonert/best_model.pt` - Best model Bài 2
- `checkpoints_viocd/history.json` - Training history Bài 1
- `checkpoints_phonert/history.json` - Training history Bài 2